0.  Setup


In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import string

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Paths & raw data


In [3]:
TRAIN_PATH = "/content/train_essays.csv"
TEST_PATH  = "/content/test_essays.csv"
PROMPT_PATH = None

src_train = pd.read_csv(TRAIN_PATH)
src_prompt = None
src_sub = pd.read_csv(TEST_PATH)

Model preparation


In [4]:
tokenizer_save_path = "bert-base-uncased"
model_save_path     = "bert-base-uncased"

tokenizer       = BertTokenizer.from_pretrained(tokenizer_save_path)
pretrained_model = BertForSequenceClassification.from_pretrained(model_save_path)
embedding_model  = pretrained_model.bert.embeddings

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters

In [5]:
train_batch_size   = 32
test_batch_size    = 32
lr                 = 1e-4
beta1              = 0.5
num_epochs         = 5
num_hidden_layers  = 3
train_ratio        = 0.8

In [9]:
"""# Data Preparation"""

class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

Data Preparation


In [10]:
all_num   = len(src_train)
train_num = int(all_num * train_ratio)
test_num  = all_num - train_num

train_set = src_train.iloc[:train_num].reset_index(drop=True)
test_set  = pd.concat([src_train.iloc[train_num:]]).reset_index(drop=True)

train_dataset = GANDAIGDataset(train_set['text'].values, train_set['generated'].values)
test_dataset  = GANDAIGDataset(test_set['text'].values, test_set['generated'].values)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size,  shuffle=False, drop_last=False)

Generator definition


In [12]:
class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 128, 4, 2, 1),
            nn.BatchNorm1d(128),
            nn.ReLU(True),
            nn.ConvTranspose1d(128, 64, 4, 2, 1),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.ConvTranspose1d(64, 768, 4, 2, 1)
        )
        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        x = self.fc(x).view(x.size(0), 256, 128)
        x = self.conv_net(x)
        x = x.transpose(1, 2)
        return self.bert_encoder(x).last_hidden_state

In [21]:
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)
        sum_mask = sum_hidden.sum(1, keepdim=True)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings

Discriminator definition


In [22]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, input):
        out = self.bert_encoder(input)
        out = self.pooler(out.last_hidden_state)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

Training utilities


In [17]:
def eval_auc(model):
    model.eval()
    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            texts, labels = batch                     # déballer
            encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
            input_ids = encodings['input_ids']
            token_type_ids = encodings['token_type_ids']
            embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
            embeded = embeded.to(device)
            label = labels.float().to(device)

            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc

Instantiate models & optimizers


In [23]:
nz = 100
config = BertConfig(num_hidden_layers=num_hidden_layers)

netG = Generator(nz)
netD = Discriminator()
criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))


netG = Generator(nz)
netD = Discriminator()

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

GAN_step call

In [24]:
optimizerG = optimizerG
optimizerD = optimizerD

Evaluate after epoch


In [26]:
def eval_auc(model):
    model.eval()
    predictions = []
    actuals = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # ajout
    with torch.no_grad():
        for batch in test_loader:
            texts, labels = batch
            encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
            input_ids = encodings['input_ids']
            token_type_ids = encodings['token_type_ids']
            embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
            embeded = embeded.to(device)
            label = labels.float().to(device)

            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc


auc_score = eval_auc(netD)


AUC: 0.2181818181818182


Inference

In [28]:


max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])

sub_dataset = InferenceDataset(src_sub['text'].values)

inference_loader = DataLoader(sub_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=lambda x: tokenizer(x, padding=True, truncation=True, max_length=256, return_tensors="pt"))

encodings = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
input_ids = encodings['input_ids']
token_type_ids = encodings['token_type_ids']
embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)

sub_ans_df = pd.DataFrame({"id": src_sub["id"], "generated": sub_predictions})

ValueError: max() arg is an empty sequence